# SimpleLLM V0.13 — GPT-style Decoder-Only Transformer

**Изменения относительно V0.12:**
- Pre-Norm архитектура (стабильнее при обучении)
- GELU активация вместо ReLU (стандарт GPT-2/3)
- Weight Tying (embedding = output projection, ~30% меньше параметров)
- Финальная LayerNorm перед выходным слоем
- Cosine Decay Learning Rate Schedule
- tf.data.Dataset pipeline с prefetch (GPU не простаивает)
- Validation split для отслеживания переобучения
- ModelCheckpoint для сохранения лучшей модели
- Безопасная детокенизация через tokenizer.detokenize()
- Бесконечный цикл генерации для тестирования

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
import keras_nlp

In [ ]:
# ========================== ГИПЕРПАРАМЕТРЫ ==========================
# Все настройки модели собраны в одном месте для удобства экспериментов

MAX_VOCAB = 15000        # Размер словаря — увеличен под расширенный корпус
CONTEXT_WIN = 50         # Длина контекстного окна (максимум токенов на вход)
EMBED_DIM = 512          # Размерность эмбеддингов (должна делиться на HEADS)
HEADS = 8                # Количество голов внимания (EMBED_DIM / HEADS = 64 на голову)
FEED_FORWARD = 2048      # Размерность FFN (стандарт: 4 × EMBED_DIM)
TRANSFORMER_BLOCKS = 4   # Количество слоёв трансформера
DROPOUT_RATE = 0.1       # Dropout для регуляризации

BATCH_SIZE = 64          # Размер батча (подбирается под GPU память)
EPOCHS = 40              # Количество эпох обучения
LEARNING_RATE = 1e-3     # Начальный learning rate
MIN_LR = 1e-5            # Минимальный learning rate (конец cosine decay)
VALIDATION_SPLIT = 0.1   # Доля данных для валидации

MAX_TRAIN_LINES = 300000 # Используем ВСЕ доступные строки (больше данных → больше словарь)
MIN_LINE_LENGTH = 15     # Минимальная длина строки (фильтрация мусора)

In [ ]:
# ========================== ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ ==========================
# Ключ к большому словарю — ЛЕКСИЧЕСКОЕ РАЗНООБРАЗИЕ, а не только объём.
# Архаичный английский (Шекспир, Библия) дают мало уникальных subwords.
# Добавляем: науку, философию, приключения, детективы, фантастику, романы.

import urllib.request, os

DATASETS = {
    # =================== ДРАМАТУРГИЯ / ПОЭЗИЯ ===================
    'shakespeare': {
        'url': 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
        'filename': 'shakespeare.txt'
    },
    'tiny_shakespeare': {
        'url': 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'filename': 'tiny_shakespeare.txt'
    },

    # =================== РЕЛИГИЯ / ФИЛОСОФИЯ ===================
    'bible_kjv': {
        'url': 'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'filename': 'bible_kjv.txt'
    },
    # Платон — "Республика" (философская лексика: justice, virtue, truth...)
    'plato_republic': {
        'url': 'https://www.gutenberg.org/cache/epub/1497/pg1497.txt',
        'filename': 'plato_republic.txt'
    },

    # =================== НАУКА ===================
    # Дарвин — "Происхождение видов" (научная лексика: species, evolution, natural selection...)
    'darwin_origin': {
        'url': 'https://www.gutenberg.org/cache/epub/1228/pg1228.txt',
        'filename': 'darwin_origin.txt'
    },
    # Эйнштейн — "Теория относительности" (физика: relativity, velocity, electromagnetic...)
    'einstein_relativity': {
        'url': 'https://www.gutenberg.org/cache/epub/5001/pg5001.txt',
        'filename': 'einstein_relativity.txt'
    },

    # =================== ГОТИКА / ХОРРОР ===================
    'poe': {
        'url': 'https://www.gutenberg.org/cache/epub/2147/pg2147.txt',
        'filename': 'poe.txt'
    },
    'frankenstein': {
        'url': 'https://www.gutenberg.org/cache/epub/84/pg84.txt',
        'filename': 'frankenstein.txt'
    },
    'dracula': {
        'url': 'https://www.gutenberg.org/cache/epub/345/pg345.txt',
        'filename': 'dracula.txt'
    },

    # =================== ДЕТЕКТИВЫ ===================
    'sherlock': {
        'url': 'https://www.gutenberg.org/cache/epub/1661/pg1661.txt',
        'filename': 'sherlock.txt'
    },

    # =================== ПРИКЛЮЧЕНИЯ / ФАНТАСТИКА ===================
    'alice': {
        'url': 'https://www.gutenberg.org/cache/epub/11/pg11.txt',
        'filename': 'alice.txt'
    },
    # Жюль Верн — "Вокруг света за 80 дней" (путешествия, география)
    'around_the_world': {
        'url': 'https://www.gutenberg.org/cache/epub/103/pg103.txt',
        'filename': 'around_the_world.txt'
    },
    # Герберт Уэллс — "Война миров" (sci-fi лексика: Martians, cylinder, heat-ray...)
    'war_of_worlds': {
        'url': 'https://www.gutenberg.org/cache/epub/36/pg36.txt',
        'filename': 'war_of_worlds.txt'
    },
    # Герберт Уэллс — "Машина времени"
    'time_machine': {
        'url': 'https://www.gutenberg.org/cache/epub/35/pg35.txt',
        'filename': 'time_machine.txt'
    },
    # Роберт Льюис Стивенсон — "Остров сокровищ" (морская лексика)
    'treasure_island': {
        'url': 'https://www.gutenberg.org/cache/epub/120/pg120.txt',
        'filename': 'treasure_island.txt'
    },
    # "20 000 лье под водой" — Жюль Верн (подводный мир, техника)
    'twenty_thousand_leagues': {
        'url': 'https://www.gutenberg.org/cache/epub/164/pg164.txt',
        'filename': 'twenty_thousand_leagues.txt'
    },

    # =================== РОМАНЫ / ПРОЗА ===================
    # Джейн Остин — "Гордость и предубеждение" (светская лексика, диалоги)
    'pride_prejudice': {
        'url': 'https://www.gutenberg.org/cache/epub/1342/pg1342.txt',
        'filename': 'pride_prejudice.txt'
    },
    # Герман Мелвилл — "Моби Дик" (морской/китобойный словарь, ~215K слов)
    'moby_dick': {
        'url': 'https://www.gutenberg.org/cache/epub/2701/pg2701.txt',
        'filename': 'moby_dick.txt'
    },
    # Марк Твен — "Приключения Тома Сойера" (разговорный американский английский)
    'tom_sawyer': {
        'url': 'https://www.gutenberg.org/cache/epub/74/pg74.txt',
        'filename': 'tom_sawyer.txt'
    },
    # Марк Твен — "Приключения Гекльберри Финна" (диалекты, сленг)
    'huck_finn': {
        'url': 'https://www.gutenberg.org/cache/epub/76/pg76.txt',
        'filename': 'huck_finn.txt'
    },
    # Чарльз Диккенс — "Большие надежды" (викторианский английский)
    'great_expectations': {
        'url': 'https://www.gutenberg.org/cache/epub/1400/pg1400.txt',
        'filename': 'great_expectations.txt'
    },
    # Чарльз Диккенс — "Повесть о двух городах"
    'tale_two_cities': {
        'url': 'https://www.gutenberg.org/cache/epub/98/pg98.txt',
        'filename': 'tale_two_cities.txt'
    },
    # Оскар Уайльд — "Портрет Дориана Грея" (эстетическая лексика)
    'dorian_gray': {
        'url': 'https://www.gutenberg.org/cache/epub/174/pg174.txt',
        'filename': 'dorian_gray.txt'
    },
    # Джозеф Конрад — "Сердце тьмы" (колониальная лексика)
    'heart_of_darkness': {
        'url': 'https://www.gutenberg.org/cache/epub/219/pg219.txt',
        'filename': 'heart_of_darkness.txt'
    },

    # =================== ПОЛИТИКА / ИСТОРИЯ ===================
    # Макиавелли — "Государь" (политическая лексика)
    'the_prince': {
        'url': 'https://www.gutenberg.org/cache/epub/1232/pg1232.txt',
        'filename': 'the_prince.txt'
    },
    # "Искусство войны" — Сунь-Цзы (военная стратегия)
    'art_of_war': {
        'url': 'https://www.gutenberg.org/cache/epub/132/pg132.txt',
        'filename': 'art_of_war.txt'
    },

    # =================== УТОПИЯ / АНТИУТОПИЯ ===================
    # Томас Мор — "Утопия"
    'utopia': {
        'url': 'https://www.gutenberg.org/cache/epub/2130/pg2130.txt',
        'filename': 'utopia.txt'
    },
}

# Директория для кэширования скачанных файлов
cache_dir = os.path.join(os.path.expanduser('~'), '.keras', 'datasets', 'llm_corpus')
os.makedirs(cache_dir, exist_ok=True)

all_lines = []
dataset_stats = {}

for name, info in DATASETS.items():
    filepath = os.path.join(cache_dir, info['filename'])
    try:
        # Скачиваем файл только если его ещё нет в кэше
        if not os.path.exists(filepath):
            print(f"  Скачиваю {name}...")
            urllib.request.urlretrieve(info['url'], filepath)

        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.read().split('\n')

        # Фильтруем строки: убираем короткие и Gutenberg-заголовки
        filtered = [
            line.strip() for line in lines
            if len(line.strip()) > MIN_LINE_LENGTH
            and not line.strip().startswith('***')               # Gutenberg header/footer
            and 'gutenberg' not in line.strip().lower()          # Gutenberg mentions
            and 'project gutenberg' not in line.strip().lower()  # Gutenberg mentions
        ]

        all_lines.extend(filtered)
        dataset_stats[name] = len(filtered)
        print(f"  ✓ {name}: {len(filtered):,} строк")

    except Exception as e:
        print(f"  ✗ {name}: не удалось загрузить — {e}")

# Перемешиваем, чтобы модель не училась порядку датасетов
np.random.shuffle(all_lines)

# Добавляем токен конца последовательности и обрезаем до лимита
train_text = [line + ' endseq' for line in all_lines[:MAX_TRAIN_LINES]]

print(f"\n{'='*60}")
print(f"  Всего строк в корпусе:    {len(all_lines):,}")
print(f"  Строк для обучения:       {len(train_text):,}")
print(f"  Датасетов загружено:      {len(dataset_stats)}/{len(DATASETS)}")
print(f"  Жанры: драма, наука, готика, детектив, sci-fi, роман,")
print(f"         философия, политика, приключения")
print(f"{'='*60}")
print(f"  Пример: {train_text[0][:100]}...")

# ========================== ТОКЕНИЗАЦИЯ (WordPiece) ==========================

# Обучаем WordPiece словарь на расширенном корпусе
# Разнообразие жанров → больше уникальных subword-единиц → словарь ближе к MAX_VOCAB
vocab_data = tf.data.Dataset.from_tensor_slices(train_text)

vocab = keras_nlp.tokenizers.compute_word_piece_vocabulary(
    vocab_data,
    vocabulary_size=MAX_VOCAB,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[START]", "[END]"]
)

# Создаём токенизатор с фиксированной длиной (CONTEXT_WIN + 1 для сдвига)
tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=CONTEXT_WIN + 1,
)

actual_vocab_size = tokenizer.vocabulary_size()
print(f"\nРеальный размер словаря: {actual_vocab_size:,} (запрошено: {MAX_VOCAB:,})")
print(f"Покрытие: {actual_vocab_size / MAX_VOCAB * 100:.1f}%")

# ========================== СОЗДАНИЕ tf.data PIPELINE ==========================
# GPU получает данные без простоев благодаря prefetch.

def tokenize_and_split(text):
    """Токенизирует текст и разделяет на input (X) и target (y) со сдвигом на 1."""
    seq = tokenizer(text)
    return seq[:-1], seq[1:]

# Токенизируем весь текст
all_tokens = tokenizer(train_text)
X_all = all_tokens[:, :-1]
y_all = all_tokens[:, 1:]

# Разделяем на train / validation
split_idx = int(len(X_all) * (1 - VALIDATION_SPLIT))

X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_val, y_val = X_all[split_idx:], y_all[split_idx:]

# Оборачиваем в tf.data.Dataset с батчированием и prefetch
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=min(len(X_train), 50000))  # Ограничиваем буфер для экономии RAM
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)  # GPU не ждёт CPU
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"Train: {len(X_train):,} samples | Validation: {len(X_val):,} samples")

In [ ]:
# ========================== МЕТРИКА ==========================

def perplexity(y_true, y_pred):
    """Perplexity — экспонента от средней кросс-энтропии.
    Чем ниже, тем лучше модель предсказывает следующий токен.
    Идеальная модель: perplexity = 1. Случайная (vocab 10k): ~10000."""
    return tf.exp(tf.reduce_mean(
        losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    ))


# ========================== TOKEN + POSITION EMBEDDING ==========================

class TokenPositionEmbedding(layers.Layer):
    """Сумма токенового и позиционного эмбеддингов.
    Позиционный эмбеддинг — обучаемый (как в GPT-2), а не синусоидальный."""

    def __init__(self, context_win, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embed = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embed = layers.Embedding(input_dim=context_win, output_dim=embed_dim)
        self.embed_dim = embed_dim

    def call(self, x):
        seq_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        # Масштабирование эмбеддингов (стабилизирует градиенты на ранних этапах)
        token_emb = self.token_embed(x) * tf.math.sqrt(tf.cast(self.embed_dim, tf.float32))
        return token_emb + self.position_embed(positions)

In [ ]:
# ========================== TRANSFORMER BLOCK (Pre-Norm) ==========================

class TransformerBlock(layers.Layer):
    """Один блок трансформера с Pre-Norm архитектурой.

    Pre-Norm (GPT-2+): LayerNorm → Attention → Residual
    Преимущество: более стабильные градиенты, можно обучать глубокие модели
    без warmup.
    """

    def __init__(self, embed_dim, heads, feed_forward, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = layers.MultiHeadAttention(
            num_heads=heads, key_dim=embed_dim // heads
        )
        self.ffn = models.Sequential([
            layers.Dense(feed_forward, activation='gelu'),  # GELU — стандарт GPT-2/3
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Pre-Norm: нормализуем ДО attention (не после, как в оригинальном Transformer)
        normed = self.norm1(inputs)
        attn_output = self.attention(normed, normed, use_causal_mask=True)
        attn_output = self.drop1(attn_output, training=training)
        x = inputs + attn_output  # Residual connection

        # Pre-Norm FFN
        normed2 = self.norm2(x)
        ffn_output = self.ffn(normed2)
        ffn_output = self.drop2(ffn_output, training=training)
        return x + ffn_output  # Residual connection

In [ ]:
# ========================== МОДЕЛЬ (GPT-style LLM) ==========================

class LLM(models.Model):
    """Decoder-only Transformer с Weight Tying.

    Weight Tying: матрица входных эмбеддингов используется повторно
    в качестве выходной проекции. Это уменьшает число параметров
    и улучшает обобщающую способность (Press & Wolf, 2017).
    """

    def __init__(self, context_win, vocab_size, embed_dim, heads,
                 feed_forward, num_blocks, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_layer = TokenPositionEmbedding(context_win, vocab_size, embed_dim)
        self.blocks = [
            TransformerBlock(embed_dim, heads, feed_forward, dropout_rate)
            for _ in range(num_blocks)
        ]
        # Финальная LayerNorm после всех блоков (стандарт GPT-2)
        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        # Weight Tying: не создаём отдельный Dense, а используем embedding-матрицу

    def call(self, inputs, training=False):
        x = self.embed_layer(inputs)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.final_norm(x)

        # Weight Tying: выход = x @ Embedding^T (вместо отдельного Dense слоя)
        # Экономит vocab_size × embed_dim параметров (~5M при наших настройках)
        logits = tf.matmul(x, self.embed_layer.token_embed.embeddings, transpose_b=True)
        return logits

In [ ]:
# ========================== ФУНКЦИЯ ГЕНЕРАЦИИ ТЕКСТА ==========================

def generate(model, prompt, max_tokens=50, temperature=0.8, top_k=10, repetition_penalty=1.2):
    """Генерирует текст автогрессивно с top-k сэмплированием.

    Args:
        model: обученная модель LLM
        prompt: начальный текст (строка)
        max_tokens: максимум новых токенов для генерации
        temperature: контроль "креативности" (0.1=консервативно, 1.5=хаотично)
        top_k: выбирать только из k наиболее вероятных токенов
        repetition_penalty: штраф за повторение уже сгенерированных токенов

    Returns:
        Сгенерированный текст (строка)
    """
    # Токенизируем промпт
    tokenized = tokenizer([prompt])
    # Совместимость: результат может быть тензором или numpy-массивом
    input_ids = tokenized.numpy()[0] if hasattr(tokenized, 'numpy') else np.array(tokenized[0])
    tokens = [int(t) for t in input_ids if t != 0]  # Убираем паддинг

    for _ in range(max_tokens):
        # Берём только последние CONTEXT_WIN токенов (скользящее окно)
        context = tokens[-CONTEXT_WIN:]
        input_tensor = tf.convert_to_tensor([context])

        # Прямой проход модели (без dropout)
        logits = model(input_tensor, training=False)
        next_logits = logits[0, -1, :].numpy()

        # Штраф за повторения: снижаем вероятность уже встречавшихся токенов
        for tok_id in set(tokens):
            if next_logits[tok_id] < 0:
                next_logits[tok_id] *= repetition_penalty
            else:
                next_logits[tok_id] /= repetition_penalty

        # Температура: масштабируем логиты перед softmax
        next_logits = next_logits / (temperature + 1e-7)

        # Top-k фильтрация: оставляем только k самых вероятных токенов
        next_logits_tf = tf.convert_to_tensor(next_logits)
        top_values, top_indices = tf.math.top_k(next_logits_tf, k=top_k)
        top_probs = tf.nn.softmax(top_values).numpy()

        # Сэмплируем из top-k с учётом распределения вероятностей
        chosen_id = int(np.random.choice(top_indices.numpy(), p=top_probs))

        # Защита от генерации паддинга [PAD] (id=0)
        if chosen_id == 0 and len(top_indices.numpy()) > 1:
            chosen_id = int(top_indices.numpy()[1])

        # Детокенизация через tokenizer (совместимо с разными версиями keras-nlp)
        detok_result = tokenizer.detokenize([chosen_id])
        if hasattr(detok_result, 'numpy'):
            token_text = detok_result.numpy().decode('utf-8').strip()
        else:
            token_text = str(detok_result).strip()

        # Остановка при токене конца последовательности
        if token_text == 'endseq':
            break

        tokens.append(chosen_id)

    # Финальная детокенизация всей последовательности целиком
    final_result = tokenizer.detokenize(tokens)
    if hasattr(final_result, 'numpy'):
        generated = final_result.numpy().decode('utf-8')
    else:
        generated = str(final_result)
    return generated

In [ ]:
# ========================== ОБУЧЕНИЕ ==========================

# Создаём модель
model = LLM(
    context_win=CONTEXT_WIN,
    vocab_size=actual_vocab_size,
    embed_dim=EMBED_DIM,
    heads=HEADS,
    feed_forward=FEED_FORWARD,
    num_blocks=TRANSFORMER_BLOCKS,
    dropout_rate=DROPOUT_RATE
)

# Cosine Decay: learning rate плавно снижается от LEARNING_RATE до MIN_LR
# Это стандартный подход для трансформеров — предотвращает осцилляции на поздних эпохах
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=total_steps,
    alpha=MIN_LR
)

# Adam с параметрами из "Attention Is All You Need"
optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9
)

model.compile(
    optimizer=optimizer,
    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[perplexity]
)

# Callbacks
model_callbacks = [
    # Сохраняем лучшую модель по val_loss (можно восстановить после обучения)
    callbacks.ModelCheckpoint(
        filepath='best_model.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    # Ранняя остановка: если val_loss не улучшается 5 эпох подряд — стоп
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# Запуск обучения
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=model_callbacks,
    verbose=1
)

print("\nОбучение завершено!")
print(f"Лучший val_loss: {min(history.history['val_loss']):.4f}")
print(f"Лучший val_perplexity: {min(history.history['val_perplexity']):.2f}")

In [ ]:
# ========================== БЕСКОНЕЧНЫЙ ЦИКЛ ГЕНЕРАЦИИ ==========================
# Введите промпт и получите сгенерированный текст.
# Для выхода введите 'exit', 'quit' или 'q'.

print("=" * 60)
print("  SimpleLLM V0.13 — Интерактивная генерация текста")
print("  Команды: 'exit'/'quit'/'q' — выход")
print("           'settings' — изменить параметры генерации")
print("=" * 60)

# Настройки генерации по умолчанию (можно менять на лету)
gen_settings = {
    'max_tokens': 50,
    'temperature': 0.8,
    'top_k': 10,
    'repetition_penalty': 1.2
}

while True:
    prompt = input("\n[Prompt] > ").strip()

    # Выход из цикла
    if prompt.lower() in ('exit', 'quit', 'q', ''):
        print("Генерация завершена.")
        break

    # Интерактивная настройка параметров генерации
    if prompt.lower() == 'settings':
        print(f"\nТекущие настройки: {gen_settings}")
        try:
            gen_settings['max_tokens'] = int(input(f"  max_tokens [{gen_settings['max_tokens']}]: ") or gen_settings['max_tokens'])
            gen_settings['temperature'] = float(input(f"  temperature [{gen_settings['temperature']}]: ") or gen_settings['temperature'])
            gen_settings['top_k'] = int(input(f"  top_k [{gen_settings['top_k']}]: ") or gen_settings['top_k'])
            gen_settings['repetition_penalty'] = float(input(f"  repetition_penalty [{gen_settings['repetition_penalty']}]: ") or gen_settings['repetition_penalty'])
            print(f"  Обновлено: {gen_settings}")
        except ValueError:
            print("  Ошибка ввода, настройки не изменены.")
        continue

    # Генерация
    result = generate(
        model, prompt,
        max_tokens=gen_settings['max_tokens'],
        temperature=gen_settings['temperature'],
        top_k=gen_settings['top_k'],
        repetition_penalty=gen_settings['repetition_penalty']
    )

    print(f"\n[Generated] {result}")